# Explicit-formula reference diagnostics

This notebook treats the exact monomial formulas for the probability-orthonormal
Legendre and first-kind Chebyshev bases as the reference. The coefficient table
is a fixed fixture generated independently with SymPy and checked against SymPy
in the test suite. Its rational coefficients are evaluated with `mpmath` at
`MP_DPS` decimal digits and rounded
once to NumPy float64. The following floating-point evaluations are compared
against that reference:

- direct evaluation of the monomial formula in `DTYPE`;
- the standard three-term recurrence in `DTYPE`;
- the ReLU, tanh, and RePU-2 root-factorized neural emulators.

The experiments resolve error by univariate degree and then sweep multivariate
basis order and dimension for either a total-degree or hyperbolic-cross index set.


In [ ]:
import json
import sys
from fractions import Fraction
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import mpmath as mp
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from neural_polynomial_emulation import (
    MultivariateBasisEmulator,
    PolynomialFormula,
    exact_multivariate_basis,
    multi_index_set,
)

FAMILIES = ("legendre", "chebyshev")
PRODUCTS = ("relu", "tanh", "repu2")
FORMULA_MAX_ORDER = 10
UNIVARIATE_MAX_ORDER = 14
ORDER_SWEEP = (1, 2, 3, 4, 5, 6, 8)
ORDER_SWEEP_DIMENSION = 2
DIMENSION_SWEEP = (1, 2, 3, 4, 5, 6)
DIMENSION_SWEEP_ORDER = 3
INDEX_SET_TYPE = "total_degree"
UNIVARIATE_POINTS = 401
MULTIVARIATE_POINTS = 128
MP_DPS = 80
DTYPE = torch.float64
RELU_DEPTH = 12
TANH_TOLERANCE = 1e-6
RANDOM_SEED = 20250918
PLOT_FLOOR = 1e-18

if DTYPE not in (torch.float32, torch.float64):
    raise ValueError("DTYPE must be torch.float32 or torch.float64")
NUMPY_DTYPE = np.float32 if DTYPE == torch.float32 else np.float64

REFERENCE_DATA_PATH = (
    ROOT / "notebooks" / "data" / "orthogonal_polynomial_coefficients.json"
)
with REFERENCE_DATA_PATH.open(encoding="utf-8") as reference_file:
    REFERENCE_DATA = json.load(reference_file)

required_reference_order = max(
    FORMULA_MAX_ORDER,
    UNIVARIATE_MAX_ORDER,
    max(ORDER_SWEEP),
    DIMENSION_SWEEP_ORDER,
)
if required_reference_order > REFERENCE_DATA["max_degree"]:
    raise ValueError(
        "requested order exceeds the fixed coefficient reference: "
        f"{required_reference_order} > {REFERENCE_DATA['max_degree']}"
    )


def reference_formula(family, degree):
    entry = REFERENCE_DATA[family][degree]
    return PolynomialFormula(
        family=family,
        degree=degree,
        coefficients=tuple(map(Fraction, entry["coefficients"])),
        normalization_radicand=entry["normalization_radicand"],
    )


## Closed-form coefficient representations

For the Legendre family,

```math
P_n(x)=2^{-n}\sum_{k=0}^{\lfloor n/2\rfloor}
(-1)^k\frac{(2n-2k)!}{k!(n-k)!(n-2k)!}x^{n-2k},
\qquad \psi_n(x)=\sqrt{2n+1}\,P_n(x).
```

For the first-kind Chebyshev family, $T_0(x)=1$ and, for $n\geq1$,

```math
T_n(x)=\frac{n}{2}\sum_{k=0}^{\lfloor n/2\rfloor}
(-1)^k\frac{(n-k-1)!}{k!(n-2k)!}(2x)^{n-2k},
\qquad \psi_0(x)=1,\quad \psi_n(x)=\sqrt{2}\,T_n(x).
```

Thus every formula is an exact rational polynomial multiplied by one square
root. The corresponding multivariate basis function is

```math
\Psi_\alpha(x)=\prod_{j=1}^d\psi_{\alpha_j}(x_j).
```


In [ ]:
def formula_table_markdown(family, max_order):
    rows = [
        f"### {family.title()}",
        "",
        "| degree | exact normalized formula |",
        "| ---: | :--- |",
    ]
    for degree in range(max_order + 1):
        formula = reference_formula(family, degree)
        expression = formula.latex()
        rows.append(
            rf"| {degree} | $\displaystyle \psi_{{{degree}}}(x)={expression}$ |"
        )
    return "\n".join(rows)

for family in FAMILIES:
    display(Markdown(formula_table_markdown(family, FORMULA_MAX_ORDER)))


## High-precision formula reference

The fixed coefficient fixture was generated from `sympy.legendre` and
`sympy.chebyshevt`; it does not use the package coefficient generator. The sampled
coordinates are ordinary floating-point numbers. The reference evaluates the
fixture's exact rational coefficients at those same coordinates with
`MP_DPS`-digit arithmetic, forms tensor products before rounding, and converts
the result to float64 only once. Consequently the reported differences include
floating-point evaluation error and neural multiplication error, but not uncertainty
in the reference coefficients.


In [ ]:
def mp_fraction(value):
    return mp.mpf(value.numerator) / value.denominator


def mp_formula_value(formula, x):
    argument = mp.mpf(float(x))
    value = mp.mpf(0)
    for coefficient in reversed(formula.coefficients):
        value = value * argument + mp_fraction(coefficient)
    return mp.sqrt(formula.normalization_radicand) * value


def high_precision_formula_basis(points, indices, family, dps=MP_DPS):
    points = np.asarray(points)
    indices = np.asarray(indices, dtype=np.int64)
    if points.ndim != 2 or indices.ndim != 2:
        raise ValueError("points and indices must be two-dimensional")
    if points.shape[1] != indices.shape[1]:
        raise ValueError("point and multi-index dimensions must agree")

    with mp.workdps(dps):
        coordinate_tables = []
        for coordinate in range(points.shape[1]):
            max_degree = int(np.max(indices[:, coordinate]))
            formulas = [
                reference_formula(family, degree)
                for degree in range(max_degree + 1)
            ]
            table = [
                [mp_formula_value(formula, value) for formula in formulas]
                for value in points[:, coordinate]
            ]
            coordinate_tables.append(table)

        result = np.empty((len(points), len(indices)), dtype=np.float64)
        for sample in range(len(points)):
            for column, alpha in enumerate(indices):
                value = mp.mpf(1)
                for coordinate, degree in enumerate(alpha):
                    value *= coordinate_tables[coordinate][sample][int(degree)]
                result[sample, column] = float(value)
    return result


def dtype_formula_value(values, formula, dtype):
    values = np.asarray(values, dtype=dtype)
    result = np.zeros_like(values)
    for coefficient in reversed(formula.coefficients):
        result = result * values + dtype(
            coefficient.numerator / coefficient.denominator
        )
    return dtype(np.sqrt(formula.normalization_radicand)) * result


def dtype_formula_basis(points, indices, family, dtype=NUMPY_DTYPE):
    points = np.asarray(points, dtype=dtype)
    indices = np.asarray(indices, dtype=np.int64)
    result = np.ones((len(points), len(indices)), dtype=dtype)
    for coordinate in range(points.shape[1]):
        max_degree = int(np.max(indices[:, coordinate]))
        table = np.column_stack(
            [
                dtype_formula_value(
                    points[:, coordinate],
                    reference_formula(family, degree),
                    dtype,
                )
                for degree in range(max_degree + 1)
            ]
        )
        result *= table[:, indices[:, coordinate]]
    return result


In [ ]:
METHODS = ("direct formula", "recurrence", "relu", "tanh", "repu2")
METHOD_LABELS = {
    "direct formula": f"direct monomial ({str(DTYPE).split('.')[-1]})",
    "recurrence": "three-term recurrence",
    "relu": "ReLU emulator",
    "tanh": "tanh emulator",
    "repu2": "RePU-2 emulator",
}
METHOD_COLORS = {
    "direct formula": "#9dc8dc",
    "recurrence": "#38b8dc",
    "relu": "#5558c9",
    "tanh": "#ee9732",
    "repu2": "#c7df55",
}
METHOD_MARKERS = {
    "direct formula": "o",
    "recurrence": "s",
    "relu": "^",
    "tanh": "D",
    "repu2": "v",
}
FIGURE_BACKGROUND = "#071018"
FIGURE_INK = "#dbe9f4"


def style_axis(axis):
    axis.set_facecolor(FIGURE_BACKGROUND)
    axis.tick_params(colors=FIGURE_INK)
    axis.xaxis.label.set_color(FIGURE_INK)
    axis.yaxis.label.set_color(FIGURE_INK)
    axis.title.set_color(FIGURE_INK)
    axis.grid(True, color="#527487", alpha=0.25)
    for spine in axis.spines.values():
        spine.set_color("#527487")


def add_shared_figure_header(fig, axes, title, *, wspace):
    handles, labels = axes[0].get_legend_handles_labels()
    fig.suptitle(title, color=FIGURE_INK, y=0.97)
    legend = fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.89),
        ncol=3,
        columnspacing=2.2,
        labelspacing=0.8,
        handletextpad=0.8,
        frameon=False,
    )
    for text in legend.get_texts():
        text.set_color(FIGURE_INK)
    fig.subplots_adjust(top=0.68, wspace=wspace)


def diagnostic_points(family, count, dimension, seed):
    if count < 3:
        raise ValueError("count must be at least three")
    rng = np.random.default_rng(seed)
    uniform = rng.random((count - 3, dimension))
    if family == "legendre":
        random_points = 2.0 * uniform - 1.0
    elif family == "chebyshev":
        random_points = np.cos(np.pi * uniform)
    else:
        raise ValueError("unknown family")
    stress_points = np.vstack(
        [
            -np.ones((1, dimension)),
            np.zeros((1, dimension)),
            np.ones((1, dimension)),
        ]
    )
    return np.vstack((stress_points, random_points))


def evaluate_all_methods(points, indices, family):
    gold = high_precision_formula_basis(points, indices, family)
    tensor_points = torch.as_tensor(points, dtype=DTYPE)
    values = {
        "direct formula": dtype_formula_basis(points, indices, family),
        "recurrence": exact_multivariate_basis(
            tensor_points, indices, family=family
        ).detach().cpu().numpy(),
    }
    for product in PRODUCTS:
        model = MultivariateBasisEmulator(
            indices,
            family=family,
            product=product,
            num_layers=RELU_DEPTH,
            tanh_tolerance=TANH_TOLERANCE,
        ).to(dtype=DTYPE)
        with torch.no_grad():
            values[product] = model(tensor_points).cpu().numpy()
    return gold, values


def error_metrics(approximation, gold):
    difference = np.asarray(approximation, dtype=np.float64) - gold
    return {
        "max_absolute_error": float(np.max(np.abs(difference))),
        "relative_frobenius_error": float(
            np.linalg.norm(difference) / np.linalg.norm(gold)
        ),
    }


def plot_error_sweep(frame, x_column, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
    fig.patch.set_facecolor(FIGURE_BACKGROUND)
    metrics = ("max_absolute_error", "relative_frobenius_error")
    titles = ("maximum absolute error", "relative Frobenius error")
    for axis, metric, metric_title in zip(axes, metrics, titles, strict=True):
        for method in METHODS:
            subset = frame[frame["method"] == method]
            axis.plot(
                subset[x_column],
                np.maximum(subset[metric], PLOT_FLOOR),
                marker=METHOD_MARKERS[method],
                color=METHOD_COLORS[method],
                label=METHOD_LABELS[method],
            )
        axis.set_yscale("log")
        axis.set_xlabel(x_column.replace("_", " "))
        axis.set_ylabel(metric_title)
        axis.set_title(metric_title)
        style_axis(axis)
    add_shared_figure_header(fig, axes, title, wspace=0.28)
    plt.show()


## Error by univariate degree

The one-dimensional experiment uses a uniform grid containing both endpoints.
For each degree, the reported value is the maximum absolute error on that grid.


In [ ]:
univariate_points = np.linspace(-1.0, 1.0, UNIVARIATE_POINTS)[:, None]
univariate_indices = np.arange(UNIVARIATE_MAX_ORDER + 1, dtype=np.int64)[:, None]
univariate_results = {}

fig, axes = plt.subplots(1, len(FAMILIES), figsize=(13, 5.4), sharey=True)
fig.patch.set_facecolor(FIGURE_BACKGROUND)
for axis, family in zip(axes, FAMILIES, strict=True):
    gold, values = evaluate_all_methods(
        univariate_points, univariate_indices, family
    )
    univariate_results[family] = (gold, values)
    for method in METHODS:
        per_degree = np.max(np.abs(values[method] - gold), axis=0)
        axis.plot(
            np.arange(UNIVARIATE_MAX_ORDER + 1),
            np.maximum(per_degree, PLOT_FLOOR),
            marker=METHOD_MARKERS[method],
            color=METHOD_COLORS[method],
            label=METHOD_LABELS[method],
        )
    axis.set_yscale("log")
    axis.set_xlabel("degree")
    axis.set_title(f"{family.title()} basis")
    style_axis(axis)
axes[0].set_ylabel("grid maximum absolute error")
add_shared_figure_header(
    fig,
    axes,
    f"Explicit-formula reference errors ({str(DTYPE).split('.')[-1]})",
    wspace=0.12,
)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(FAMILIES), figsize=(13, 5.4), sharey=False)
fig.patch.set_facecolor(FIGURE_BACKGROUND)
degree = UNIVARIATE_MAX_ORDER
for axis, family in zip(axes, FAMILIES, strict=True):
    gold, values = univariate_results[family]
    for method in METHODS:
        pointwise = np.abs(values[method][:, degree] - gold[:, degree])
        axis.plot(
            univariate_points[:, 0],
            np.maximum(pointwise, PLOT_FLOOR),
            color=METHOD_COLORS[method],
            label=METHOD_LABELS[method],
        )
    axis.set_yscale("log")
    axis.set_xlabel("x")
    axis.set_ylabel("absolute error")
    axis.set_title(f"{family.title()}: degree {degree}")
    style_axis(axis)
add_shared_figure_header(
    fig, axes, "Pointwise error of the highest displayed mode", wspace=0.28
)
plt.show()


## Increasing multivariate basis order

The dimension is fixed at `ORDER_SWEEP_DIMENSION`. Random points follow the
family's product probability measure: uniform coordinates for Legendre and
arcsine-distributed coordinates for Chebyshev. The three diagonal stress points
$(-1,\ldots,-1)$, $0$, and $(1,\ldots,1)$ are included because monomial
cancellation and multiplication error can be largest near the boundary.


In [ ]:
order_rows = []
for family_index, family in enumerate(FAMILIES):
    points = diagnostic_points(
        family,
        MULTIVARIATE_POINTS,
        ORDER_SWEEP_DIMENSION,
        RANDOM_SEED + family_index,
    )
    for order in ORDER_SWEEP:
        indices = multi_index_set(
            ORDER_SWEEP_DIMENSION, order, INDEX_SET_TYPE
        )
        gold, values = evaluate_all_methods(points, indices, family)
        for method in METHODS:
            order_rows.append(
                {
                    "family": family,
                    "dimension": ORDER_SWEEP_DIMENSION,
                    "order": order,
                    "basis_size": len(indices),
                    "method": method,
                    **error_metrics(values[method], gold),
                }
            )
order_errors = pd.DataFrame(order_rows)

for family in FAMILIES:
    plot_error_sweep(
        order_errors[order_errors["family"] == family],
        "order",
        (
            f"{family.title()}, d={ORDER_SWEEP_DIMENSION}, "
            f"{INDEX_SET_TYPE.replace('_', ' ')} index set"
        ),
    )

display(order_errors)


## Increasing dimension

The order is fixed at `DIMENSION_SWEEP_ORDER`. A single maximum-dimensional
point set is generated for each family and truncated to its first $d$ coordinates,
so the sampled inputs are nested across the dimension sweep.


In [ ]:
dimension_rows = []
maximum_dimension = max(DIMENSION_SWEEP)
for family_index, family in enumerate(FAMILIES):
    maximum_points = diagnostic_points(
        family,
        MULTIVARIATE_POINTS,
        maximum_dimension,
        RANDOM_SEED + 100 + family_index,
    )
    for dimension in DIMENSION_SWEEP:
        points = maximum_points[:, :dimension]
        indices = multi_index_set(
            dimension, DIMENSION_SWEEP_ORDER, INDEX_SET_TYPE
        )
        gold, values = evaluate_all_methods(points, indices, family)
        for method in METHODS:
            dimension_rows.append(
                {
                    "family": family,
                    "dimension": dimension,
                    "order": DIMENSION_SWEEP_ORDER,
                    "basis_size": len(indices),
                    "method": method,
                    **error_metrics(values[method], gold),
                }
            )
dimension_errors = pd.DataFrame(dimension_rows)

for family in FAMILIES:
    plot_error_sweep(
        dimension_errors[dimension_errors["family"] == family],
        "dimension",
        (
            f"{family.title()}, order={DIMENSION_SWEEP_ORDER}, "
            f"{INDEX_SET_TYPE.replace('_', ' ')} index set"
        ),
    )

display(dimension_errors)


## Interpretation

The high-precision monomial formula is the diagnostic reference, not a recommended
production evaluator. Ordinary floating-point monomial evaluation can lose accuracy
through cancellation as degree increases. The recurrence avoids much of that
cancellation. RePU-2 differs from the exact polynomial only through root computation,
factor ordering, and floating-point arithmetic; ReLU and tanh additionally contain
approximate multipliers. Multivariate errors combine univariate evaluation error with
the cross-coordinate product chain.

All maxima are over the displayed finite grid or sampled point set. They are not
certified uniform-error bounds.
